# 📋 Reconciliation Template SNOW Data Populator

## What this notebook does

For each Excel reconciliation template:

1. Reads **`General Model & Review Summary`** tab:
   - `B4` → Model ID
   - `E4` → Use Status (`Model In-Use` / `Model Not In-Use`)
   - Updates `G4` and `I10` with user-supplied dates (mm/dd/yyyy)
   - Clears `F10`

2. Looks up the Model ID in the **SNOW dump** (Excel/CSV)

3. Populates **`Inventory Recon & Evidence`** tab columns `D7:D41`:

| Scenario | Condition | Action |
|---|---|---|
| **In-Use — D20/21/22/26/27/40** | Both blank/N/A | Skip |
| **In-Use — D20/21/22/26/27/40** | SNOW has value, cell blank/N/A | Replace + 🟡 Yellow |
| **In-Use — all other cells** | Values match | Skip |
| **In-Use — all other cells** | Values differ | Replace with SNOW + 🟡 Yellow |
| **In-Use — all other cells** | SNOW empty, cell has data | Write `Blank` + 🔴 Red |
| **Not In-Use** | D7:D14 only — SNOW blank/N/A | Replace with SNOW + 🔴 Red |
| **Not In-Use** | D7:D14 — values differ | Replace with SNOW + 🟡 Yellow |
| **Not In-Use** | D7:D14 — values match | Skip |


---
## Cell 1 — Install & Import Libraries

In [ ]:
import subprocess, sys

for pkg in ["pandas", "openpyxl"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ Libraries ready.")

In [ ]:
import os
import re
import logging
import traceback
import copy
from pathlib import Path
from datetime import datetime

import pandas as pd
import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

print(f"pandas   : {pd.__version__}")
print(f"openpyxl : {openpyxl.__version__}")
print("✅ Imports OK.")

---
## Cell 2 — ✏️ Configuration  *(edit before running)*

In [ ]:
# ──────────────────────────────────────────────────────────────────
#  PATHS
# ──────────────────────────────────────────────────────────────────

# Folder containing all reconciliation Excel templates
TEMPLATES_FOLDER = r"C:\reconciliation_templates"      # ← CHANGE

# SNOW dump file — Excel or CSV with one row per model
# Must have a column matching MODEL_ID_COL and columns matching SNOW_FIELD_MAP values
SNOW_DUMP_FILE   = r"C:\snow_dump\snow_data.xlsx"      # ← CHANGE

# Folder where updated template copies will be saved (originals are NOT overwritten)
OUTPUT_FOLDER    = r"C:\reconciliation_output_updated" # ← CHANGE

# ──────────────────────────────────────────────────────────────────
#  DATE INPUTS  (mm/dd/yyyy)
# ──────────────────────────────────────────────────────────────────
DATE_G4  = "06/03/2026"   # Written to General Model & Review Summary → G4
DATE_I10 = "06/03/2026"   # Written to General Model & Review Summary → I10

# ──────────────────────────────────────────────────────────────────
#  SNOW DUMP COLUMN MAPPING
#  Key   = Column header in your SNOW dump spreadsheet
#  Value = Not used for lookup; order maps to D7..D41 rows
#
#  IMPORTANT: The dict below maps each D-row (7 to 41) to the
#  corresponding column name in your SNOW dump.
#  Adjust column names to match your actual SNOW dump headers.
# ──────────────────────────────────────────────────────────────────
MODEL_ID_COL = "Model_ID"   # Column in SNOW dump that holds the Model ID (matches B4)

# Row number → SNOW dump column name
# Add/remove rows as needed; must cover rows 7–41
ROW_TO_SNOW_COL = {
    7:  "MRM_Inventory_Name",
    8:  "Business_Owner",
    9:  "Developer",
    10: "Model_User",
    11: "Use_Status",
    12: "Line_of_Business",
    13: "Purpose_and_Description",
    14: "Development_Group",
    15: "Validation_Lead",
    16: "AI_ML_Flag",
    17: "Tier",
    18: "Model_Type",
    19: "Model_Sub_Type",
    20: "Risk_Rating",           # optional / can be blank
    21: "Risk_Rating_Date",      # optional / can be blank
    22: "Risk_Rating_Rationale", # optional / can be blank
    23: "Last_Validation_Date",
    24: "Next_Validation_Date",
    25: "Validation_Type",
    26: "Regulatory_Classification",  # optional / can be blank
    27: "Regulatory_Applicability",   # optional / can be blank
    28: "Model_Status",
    29: "Change_Type",
    30: "Change_Description",
    31: "Change_Effective_Date",
    32: "Approval_Date",
    33: "Model_Version",
    34: "Implementation_Date",
    35: "Retirement_Date",
    36: "Retirement_Reason",
    37: "Related_Models",
    38: "Data_Sources",
    39: "Output_Type",
    40: "Materiality",           # optional / can be blank
    41: "Additional_Notes",
}

# ──────────────────────────────────────────────────────────────────
#  ROWS THAT ARE ALLOWED TO BE BLANK (In-Use models only)
# ──────────────────────────────────────────────────────────────────
OPTIONAL_ROWS_IN_USE = {20, 21, 22, 26, 27, 40}

# ──────────────────────────────────────────────────────────────────
#  SHEET NAMES
# ──────────────────────────────────────────────────────────────────
SUMMARY_SHEET_CANDIDATES = [
    "General Model & Review Summary",
    "General Model Review Summary",
    "Model Summary",
    "Summary",
]
RECON_SHEET_CANDIDATES = [
    "Inventory Recon & Evidence",
    "Inventory Recon",
    "Recon",
]

# ──────────────────────────────────────────────────────────────────
#  COLOURS
# ──────────────────────────────────────────────────────────────────
YELLOW_FILL = PatternFill("solid", fgColor="FFFF00")
RED_FILL    = PatternFill("solid", fgColor="FF0000")
NO_FILL     = PatternFill(fill_type=None)

# ──────────────────────────────────────────────────────────────────
#  CREATE OUTPUT FOLDER
# ──────────────────────────────────────────────────────────────────
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
LOG_FILE = os.path.join(OUTPUT_FOLDER, "populator_log.txt")

print("✅ Configuration loaded.")
print(f"   Templates : {TEMPLATES_FOLDER}")
print(f"   SNOW dump : {SNOW_DUMP_FILE}")
print(f"   Output    : {OUTPUT_FOLDER}")
print(f"   Date G4   : {DATE_G4}")
print(f"   Date I10  : {DATE_I10}")

---
## Cell 3 — Logging Setup

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ]
)
logger = logging.getLogger("recon_populator")
logger.info("Logging started → %s", LOG_FILE)

---
## Cell 4 — Helper Functions

In [ ]:
# ── Value normalisation ────────────────────────────────────────────

def norm(value) -> str:
    """Strip, upper-case, and return string; treat None / 'N/A' / 'NA' as empty."""
    if value is None:
        return ""
    s = str(value).strip()
    if s.upper() in ("", "N/A", "NA", "NONE", "-"):
        return ""
    return s


def is_blank(value) -> bool:
    return norm(value) == ""


def validate_date(date_str: str) -> str:
    """Validate mm/dd/yyyy format; raise ValueError if invalid."""
    try:
        datetime.strptime(date_str, "%m/%d/%Y")
    except ValueError:
        raise ValueError(f"Date '{date_str}' is not in mm/dd/yyyy format.")
    return date_str


# ── Sheet finder ───────────────────────────────────────────────────

def find_sheet(wb: openpyxl.Workbook, candidates: list) -> openpyxl.worksheet.worksheet.Worksheet:
    """Return first sheet whose name matches any candidate (case-insensitive)."""
    lower_map = {s.lower(): s for s in wb.sheetnames}
    for c in candidates:
        if c.lower() in lower_map:
            return wb[lower_map[c.lower()]]
    raise KeyError(
        f"None of {candidates} found in workbook. Available: {wb.sheetnames}"
    )


# ── Cell fill helper ───────────────────────────────────────────────

def apply_fill(cell, fill: PatternFill):
    """Apply a fill while preserving other cell formatting."""
    cell.fill = fill


# ── SNOW lookup ────────────────────────────────────────────────────

def build_snow_index(snow_df: pd.DataFrame) -> dict:
    """
    Build {model_id (str, stripped, upper): row_series} dict from SNOW dump.
    MODEL_ID_COL must exist in snow_df.
    """
    if MODEL_ID_COL not in snow_df.columns:
        raise KeyError(
            f"Column '{MODEL_ID_COL}' not found in SNOW dump. "
            f"Available columns: {list(snow_df.columns)}"
        )
    index = {}
    for _, row in snow_df.iterrows():
        mid = norm(row[MODEL_ID_COL]).upper()
        if mid:
            index[mid] = row
    return index


def get_snow_value(snow_row, row_num: int) -> str:
    """Return normalised SNOW value for a given template row number."""
    col = ROW_TO_SNOW_COL.get(row_num)
    if col is None or col not in snow_row.index:
        return ""   # no mapping defined → treat as empty
    return norm(snow_row[col])


print("✅ Helper functions defined.")

---
## Cell 5 — Load SNOW Dump

In [ ]:
snow_ext = Path(SNOW_DUMP_FILE).suffix.lower()

if snow_ext in (".xlsx", ".xls"):
    snow_df = pd.read_excel(SNOW_DUMP_FILE, dtype=str)
elif snow_ext == ".csv":
    snow_df = pd.read_csv(SNOW_DUMP_FILE, dtype=str)
else:
    raise ValueError(f"Unsupported SNOW dump format: {snow_ext}")

# Strip whitespace from all string columns
snow_df = snow_df.apply(lambda col: col.str.strip() if col.dtype == object else col)

print(f"SNOW dump loaded: {snow_df.shape[0]} rows × {snow_df.shape[1]} columns")
print(f"Columns: {list(snow_df.columns)}")

# Build fast lookup index
snow_index = build_snow_index(snow_df)
print(f"\n✅ SNOW index built: {len(snow_index)} unique model IDs.")

---
## Cell 6 — Validate Date Inputs

In [ ]:
validate_date(DATE_G4)
validate_date(DATE_I10)
print(f"✅ Dates valid:  G4 = {DATE_G4}   I10 = {DATE_I10}")

---
## Cell 7 — Core Processing Function

In [ ]:
def process_template(src_path: str, out_path: str, snow_index: dict) -> dict:
    """
    Process one reconciliation template:
      1. Update General Model & Review Summary (G4, I10, clear F10)
      2. Read Model ID (B4) and Use Status (E4)
      3. Apply SNOW data rules to Inventory Recon & Evidence (D7:D41)

    Returns a summary dict for reporting.
    """
    result = {
        "file"            : Path(src_path).name,
        "model_id"        : "UNKNOWN",
        "use_status"      : "UNKNOWN",
        "snow_found"      : False,
        "cells_changed"   : 0,
        "cells_yellow"    : 0,
        "cells_red"       : 0,
        "status"          : "OK",
        "notes"           : [],
    }

    # ── Load workbook (keep formulas; do NOT use data_only) ────────
    wb = load_workbook(src_path)

    # ── Step 1: General Model & Review Summary ─────────────────────
    try:
        ws_summary = find_sheet(wb, SUMMARY_SHEET_CANDIDATES)
    except KeyError as e:
        result["status"] = "ERROR"
        result["notes"].append(str(e))
        wb.close()
        return result

    # Read Model ID from B4
    model_id_raw = ws_summary["B4"].value
    model_id     = norm(model_id_raw).upper()
    result["model_id"] = model_id or "BLANK_B4"

    # Read Use Status from E4
    use_status_raw = norm(ws_summary["E4"].value)
    result["use_status"] = use_status_raw

    is_in_use = "not" not in use_status_raw.lower()   # True = In-Use

    if not use_status_raw:
        result["notes"].append("E4 is blank; treated as In-Use.")
        is_in_use = True

    # Write dates and clear F10
    ws_summary["G4"].value  = DATE_G4
    ws_summary["I10"].value = DATE_I10
    ws_summary["F10"].value = None            # clear content

    # ── Step 2: Look up SNOW data ──────────────────────────────────
    snow_row = snow_index.get(model_id)
    if snow_row is None:
        result["snow_found"] = False
        result["notes"].append(f"Model ID '{model_id}' NOT found in SNOW dump.")
        wb.save(out_path)
        wb.close()
        result["status"] = "WARN"
        return result

    result["snow_found"] = True

    # ── Step 3: Inventory Recon & Evidence sheet ───────────────────
    try:
        ws_recon = find_sheet(wb, RECON_SHEET_CANDIDATES)
    except KeyError as e:
        result["status"] = "ERROR"
        result["notes"].append(str(e))
        wb.close()
        return result

    # Determine which rows to process
    if is_in_use:
        rows_to_process = range(7, 42)    # D7:D41
    else:
        rows_to_process = range(7, 15)    # D7:D14 only

    for row_num in rows_to_process:
        cell       = ws_recon.cell(row=row_num, column=4)   # column D
        cell_val   = norm(cell.value)
        snow_val   = get_snow_value(snow_row, row_num)

        if is_in_use:
            # ── IN-USE LOGIC ───────────────────────────────────────

            if row_num in OPTIONAL_ROWS_IN_USE:
                # Optional rows: only act when SNOW has a value
                if is_blank(snow_val):
                    # Both blank / SNOW empty → skip entirely
                    continue
                elif is_blank(cell_val):
                    # SNOW has value, cell is blank/N/A → replace + yellow
                    cell.value = snow_val
                    apply_fill(cell, YELLOW_FILL)
                    result["cells_changed"] += 1
                    result["cells_yellow"]  += 1
                elif cell_val.lower() != snow_val.lower():
                    # Mismatch → replace + yellow
                    cell.value = snow_val
                    apply_fill(cell, YELLOW_FILL)
                    result["cells_changed"] += 1
                    result["cells_yellow"]  += 1
                # else: match → skip

            else:
                # Required rows
                if is_blank(snow_val) and not is_blank(cell_val):
                    # SNOW empty but template has data → write 'Blank' + red
                    cell.value = "Blank"
                    apply_fill(cell, RED_FILL)
                    result["cells_changed"] += 1
                    result["cells_red"]     += 1
                elif is_blank(snow_val) and is_blank(cell_val):
                    # Both empty → skip
                    continue
                elif cell_val.lower() == snow_val.lower():
                    # Match → skip
                    continue
                else:
                    # Mismatch (includes cell blank/N/A with SNOW having value)
                    cell.value = snow_val
                    apply_fill(cell, YELLOW_FILL)
                    result["cells_changed"] += 1
                    result["cells_yellow"]  += 1

        else:
            # ── NOT IN-USE LOGIC (D7:D14 only) ────────────────────
            # Preference: SNOW data always wins
            if is_blank(snow_val):
                # SNOW blank → highlight red (data should be there)
                if is_blank(cell_val):
                    # Both blank → write 'Blank' + red
                    cell.value = "Blank"
                else:
                    # Template has value but SNOW is empty → write 'Blank' + red
                    cell.value = "Blank"
                apply_fill(cell, RED_FILL)
                result["cells_changed"] += 1
                result["cells_red"]     += 1
            elif cell_val.lower() == snow_val.lower():
                # Match → skip
                continue
            else:
                # Mismatch → replace with SNOW + yellow
                cell.value = snow_val
                apply_fill(cell, YELLOW_FILL)
                result["cells_changed"] += 1
                result["cells_yellow"]  += 1

    # ── Save updated workbook ──────────────────────────────────────
    wb.save(out_path)
    wb.close()
    return result


print("✅ process_template() defined.")

---
## Cell 8 — Discover All Template Files

In [ ]:
template_files = sorted(
    str(p)
    for p in Path(TEMPLATES_FOLDER).rglob("*")
    if p.suffix.lower() in {".xlsx", ".xls"}
    and not p.name.startswith("~$")
)

print(f"📂 Templates folder : {TEMPLATES_FOLDER}")
print(f"📄 Files found      : {len(template_files)}")

if not template_files:
    print("⚠️  No files found. Check TEMPLATES_FOLDER path.")
else:
    print("\nFirst 5:")
    for f in template_files[:5]:
        print(" ", Path(f).name)

---
## Cell 9 — Process All Templates

In [ ]:
all_results   = []
failed_files  = []
processed_ok  = 0
total         = len(template_files)

for idx, src_path in enumerate(template_files, start=1):
    fname    = Path(src_path).name
    out_path = os.path.join(OUTPUT_FOLDER, fname)

    try:
        res = process_template(src_path, out_path, snow_index)
        all_results.append(res)
        processed_ok += 1

        status_icon = "✅" if res["status"] == "OK" else ("⚠️" if res["status"] == "WARN" else "❌")
        logger.info(
            "[%d/%d] %s %s | ID=%s | %s | changed=%d yellow=%d red=%d | %s",
            idx, total, status_icon, fname,
            res["model_id"], res["use_status"],
            res["cells_changed"], res["cells_yellow"], res["cells_red"],
            "; ".join(res["notes"]) if res["notes"] else "OK"
        )

    except Exception as exc:
        failed_files.append({"File": fname, "Error": str(exc)})
        logger.error("[%d/%d] ❌ EXCEPTION %s → %s", idx, total, fname, exc)
        logger.debug(traceback.format_exc())

print(f"\n{'='*55}")
print(f"  Total files   : {total}")
print(f"  Processed     : {processed_ok}")
print(f"  Hard failures : {len(failed_files)}")
print(f"{'='*55}")

---
## Cell 10 — Build Results DataFrame & Export Reports

In [ ]:
results_df = pd.DataFrame(all_results)

# Expand notes list to string
if "notes" in results_df.columns:
    results_df["notes"] = results_df["notes"].apply(lambda x: "; ".join(x) if x else "")

# ── Export processing report ───────────────────────────────────────
report_path = os.path.join(OUTPUT_FOLDER, "processing_report.xlsx")

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    results_df.to_excel(writer, index=False, sheet_name="Results")

    if failed_files:
        pd.DataFrame(failed_files).to_excel(writer, index=False, sheet_name="Failed")

    # Auto-size columns
    for sname in writer.sheets:
        ws = writer.sheets[sname]
        for col in ws.columns:
            max_w = max((len(str(c.value)) if c.value else 0) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max_w + 4, 80)

print(f"✅ Processing report saved → {report_path}")

# Preview
results_df.head(10)

---
## Cell 11 — Final Summary

In [ ]:
ok_count   = (results_df["status"] == "OK").sum()   if not results_df.empty else 0
warn_count = (results_df["status"] == "WARN").sum() if not results_df.empty else 0
err_count  = (results_df["status"] == "ERROR").sum()if not results_df.empty else 0
in_use_ct  = results_df["use_status"].str.contains("Not", case=False, na=False)

total_yellow = results_df["cells_yellow"].sum() if not results_df.empty else 0
total_red    = results_df["cells_red"].sum()    if not results_df.empty else 0
total_chg    = results_df["cells_changed"].sum()if not results_df.empty else 0

print("="*60)
print("      RECONCILIATION SNOW POPULATOR — SUMMARY")
print("="*60)
print(f"  📂  Templates found          : {total}")
print(f"  ✅  Processed OK             : {ok_count}")
print(f"  ⚠️   Warnings (no SNOW match) : {warn_count}")
print(f"  ❌  Errors                   : {err_count + len(failed_files)}")
print()
print(f"  📊  Total cells changed      : {total_chg}")
print(f"  🟡  Yellow highlights        : {total_yellow}")
print(f"  🔴  Red highlights           : {total_red}")
print()
print("  OUTPUT:")
print(f"    • Updated templates → {OUTPUT_FOLDER}/")
print(f"    • Processing report → {report_path}")
print(f"    • Log file          → {LOG_FILE}")
print("="*60)

---
## Cell 12 — (Optional) Inspect a Single File for Testing

In [ ]:
# ── Set TEST_FILE to the filename you want to inspect ─────────────
TEST_FILE = r"C:\reconciliation_templates\AA0019.xlsx"   # ← CHANGE

if Path(TEST_FILE).exists():
    TEST_OUT = os.path.join(OUTPUT_FOLDER, "TEST_" + Path(TEST_FILE).name)
    res = process_template(TEST_FILE, TEST_OUT, snow_index)
    print("\n--- Single File Test Result ---")
    for k, v in res.items():
        print(f"  {k:<20} : {v}")
    print(f"\n  ✅ Output saved → {TEST_OUT}")
else:
    print(f"⚠️  File not found: {TEST_FILE}")
    print("   Update TEST_FILE path to run this test.")